In [1]:
from pathlib import Path
import pandas as pd

# Set the data directory (CSV files are stored here)
DATA_DIR = Path("../data")

# CSV file paths
ANXIETY_FILE_1 = DATA_DIR / "enhanced_anxiety_dataset.csv"
ANXIETY_FILE_2 = DATA_DIR / "family_anxiety_14_dataset.csv"
SLEEP_FILE = DATA_DIR / "Sleep_health_and_lifestyle_dataset.csv"

print("Anxiety file 1:", ANXIETY_FILE_1)
print("Anxiety file 2:", ANXIETY_FILE_2)
print("Sleep file:    ", SLEEP_FILE)

# Quick check: load a few rows from each file
anx1_sample = pd.read_csv(ANXIETY_FILE_1).head()
anx2_sample = pd.read_csv(ANXIETY_FILE_2).head()
sleep_sample = pd.read_csv(SLEEP_FILE).head()

anx1_sample, anx2_sample, sleep_sample


Anxiety file 1: ../data/enhanced_anxiety_dataset.csv
Anxiety file 2: ../data/family_anxiety_14_dataset.csv
Sleep file:     ../data/Sleep_health_and_lifestyle_dataset.csv


(   Age  Gender Occupation  Sleep Hours  Physical Activity (hrs/week)  \
 0   29  Female     Artist          6.0                           2.7   
 1   46   Other      Nurse          6.2                           5.7   
 2   64    Male      Other          5.0                           3.7   
 3   20  Female  Scientist          5.8                           2.8   
 4   49  Female      Other          8.2                           2.3   
 
    Caffeine Intake (mg/day)  Alcohol Consumption (drinks/week) Smoking  \
 0                       181                                 10     Yes   
 1                       200                                  8     Yes   
 2                       117                                  4      No   
 3                       360                                  6     Yes   
 4                       247                                  4     Yes   
 
   Family History of Anxiety  Stress Level (1-10)  Heart Rate (bpm)  \
 0                        No         

In [4]:
def standardize_gender(series: pd.Series) -> pd.Series:
    gender = series.astype(str).str.strip().str.lower()
    mapped = gender.map({"male": "Male", "female": "Female"})
    return mapped.fillna("Other")


OCCUPATION_MAP = {
    "artist": "Artist",
    "musician": "Artist",
    "nurse": "Nurse",
    "doctor": "Doctor",
    "teacher": "Teacher",
    "scientist": "Scientist",
    "lawyer": "Lawyer",
    "student": "Student",
    "engineer": "Engineer",
    "freelancer": "Other",
    "chef": "Service",
    "athlete": "Athlete",
    "other": "Other",
    "software engineer": "Engineer",
    "sales representative": "Sales",
    "salesperson": "Sales",
    "accountant": "Accountant",
    "manager": "Manager",
}


def standardize_occupation(series: pd.Series) -> pd.Series:
    occ = series.astype(str).str.strip().str.lower()
    mapped = occ.map(OCCUPATION_MAP)
    return mapped.fillna("Other")


In [5]:
def make_age_group(series: pd.Series) -> pd.Series:
    def _age_bin(a):
        try:
            a = float(a)
        except:
            return pd.NA

        if a < 20:
            return "18-19"
        elif a < 30:
            return "20-29"
        elif a < 40:
            return "30-39"
        elif a < 50:
            return "40-49"
        elif a < 60:
            return "50-59"
        else:
            return "60-69"

    return series.apply(_age_bin)


In [6]:
def prepare_anxiety_data(df1: pd.DataFrame, df2: pd.DataFrame) -> pd.DataFrame:
    """Combine the two anxiety datasets and standardize keys."""
    # Merge the two anxiety datasets
    df = pd.concat([df1, df2], ignore_index=True)
    df = df.copy()

    # Standardize demographic keys
    df["Gender_std"] = standardize_gender(df["Gender"])
    df["Occupation_std"] = standardize_occupation(df["Occupation"])
    df["AgeGroup"] = make_age_group(df["Age"])

    # Convert numeric columns
    numeric_cols = [
        "Sleep Hours",
        "Physical Activity (hrs/week)",
        "Caffeine Intake (mg/day)",
        "Alcohol Consumption (drinks/week)",
        "Stress Level (1-10)",
        "Heart Rate (bpm)",
        "Therapy Sessions (per month)",
        "Diet Quality (1-10)",
        "Anxiety Level (1-10)",
    ]

    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    return df


# Load raw CSVs
anx1 = pd.read_csv(ANXIETY_FILE_1)
anx2 = pd.read_csv(ANXIETY_FILE_2)

# Clean and prepare the anxiety dataset
anxiety_prepared = prepare_anxiety_data(anx1, anx2)

# Show first rows
anxiety_prepared.head()


,Age,Gender,Occupation,Sleep Hours,Physical Activity (hrs/week),Caffeine Intake (mg/day),Alcohol Consumption (drinks/week),Smoking,Family History of Anxiety,Stress Level (1-10),...,Sweating Level (1-5),Dizziness,Medication,Therapy Sessions (per month),Recent Major Life Event,Diet Quality (1-10),Anxiety Level (1-10),Gender_std,Occupation_std,AgeGroup
0,29,Female,Artist,6.0,2.7,181,10,Yes,No,10,...,4,No,Yes,3,Yes,7,5.0,Female,Artist,20-29
1,46,Other,Nurse,6.2,5.7,200,8,Yes,Yes,1,...,2,Yes,No,2,No,8,3.0,Other,Nurse,40-49
2,64,Male,Other,5.0,3.7,117,4,No,Yes,1,...,3,No,No,1,Yes,1,1.0,Male,Other,60-69
3,20,Female,Scientist,5.8,2.8,360,6,Yes,No,4,...,3,No,No,0,No,1,2.0,Female,Scientist,20-29
4,49,Female,Other,8.2,2.3,247,4,Yes,No,1,...,4,Yes,Yes,1,No,3,1.0,Female,Other,40-49


In [7]:
def aggregate_anxiety(df: pd.DataFrame) -> pd.DataFrame:
    """Aggregate anxiety data by AgeGroup + Gender + Occupation."""
    df = df.copy()

    # Create binary indicators
    df["HighAnxiety"] = (df["Anxiety Level (1-10)"] >= 6).astype(int)
    df["Smoker"] = df["Smoking"].astype(str).str.strip().str.lower().eq("yes").astype(int)
    df["FamilyHistory"] = (
        df["Family History of Anxiety"]
        .astype(str).str.strip().str.lower().eq("yes").astype(int)
    )

    group_cols = ["AgeGroup", "Gender_std", "Occupation_std"]

    agg = (
        df.groupby(group_cols)
        .agg(
            anx_sleep_hours_mean=("Sleep Hours", "mean"),
            anx_physical_activity_mean=("Physical Activity (hrs/week)", "mean"),
            anx_caffeine_mean=("Caffeine Intake (mg/day)", "mean"),
            anx_alcohol_mean=("Alcohol Consumption (drinks/week)", "mean"),
            anx_stress_mean=("Stress Level (1-10)", "mean"),
            anx_heart_rate_mean=("Heart Rate (bpm)", "mean"),
            anx_diet_quality_mean=("Diet Quality (1-10)", "mean"),
            anx_anxiety_mean=("Anxiety Level (1-10)", "mean"),
            anx_high_anxiety_rate=("HighAnxiety", "mean"),
            anx_smoking_rate=("Smoker", "mean"),
            anx_family_history_rate=("FamilyHistory", "mean"),
            n_anxiety_records=("Age", "count"),
        )
        .reset_index()
    )

    # Rename standardized columns back to simple names
    agg = agg.rename(columns={"Gender_std": "Gender", "Occupation_std": "Occupation"})
    return agg


# Run aggregation
anxiety_agg = aggregate_anxiety(anxiety_prepared)
anxiety_agg.head()


,AgeGroup,Gender,Occupation,anx_sleep_hours_mean,anx_physical_activity_mean,anx_caffeine_mean,anx_alcohol_mean,anx_stress_mean,anx_heart_rate_mean,anx_diet_quality_mean,anx_anxiety_mean,anx_high_anxiety_rate,anx_smoking_rate,anx_family_history_rate,n_anxiety_records
0,18-19,Female,Artist,7.151163,3.041860,256.744186,8.883721,5.558140,90.558140,5.395349,3.465116,0.069767,0.511628,0.372093,43
1,18-19,Female,Athlete,7.338462,2.726923,237.692308,10.615385,4.846154,95.307692,4.653846,3.192308,0.038462,0.307692,0.269231,26
2,18-19,Female,Doctor,6.925000,3.241667,352.125000,10.333333,5.916667,89.958333,5.458333,3.458333,0.083333,0.333333,0.333333,24
3,18-19,Female,Engineer,7.111538,3.319231,326.153846,9.269231,4.192308,92.807692,5.692308,3.153846,0.038462,0.500000,0.307692,26
4,18-19,Female,Lawyer,7.113333,3.260000,362.000000,10.133333,4.866667,93.200000,6.400000,3.133333,0.133333,0.200000,0.333333,15


In [8]:
def prepare_sleep_data(df: pd.DataFrame) -> pd.DataFrame:
    """Clean and standardize the sleep and lifestyle dataset."""
    df = df.copy()

    # Standardize keys
    df["Gender_std"] = standardize_gender(df["Gender"])
    df["Occupation_std"] = standardize_occupation(df["Occupation"])
    df["AgeGroup"] = make_age_group(df["Age"])

    # Numeric columns
    numeric_cols = [
        "Sleep Duration",
        "Quality of Sleep",
        "Physical Activity Level",
        "Stress Level",
        "Heart Rate",
        "Daily Steps",
    ]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Sleep disorder indicator
    df["HasSleepDisorder"] = (
        df["Sleep Disorder"].astype(str).str.strip().str.lower().ne("none").astype(int)
    )

    return df


# Apply cleaning
sleep_raw = pd.read_csv(SLEEP_FILE)
sleep_prepared = prepare_sleep_data(sleep_raw)

sleep_prepared.head()


,Person ID,Gender,Age,Occupation,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,BMI Category,Blood Pressure,Heart Rate,Daily Steps,Sleep Disorder,Gender_std,Occupation_std,AgeGroup,HasSleepDisorder
0,1,Male,27,Software Engineer,6.1,6,42,6,Overweight,126/83,77,4200,NaN,Male,Engineer,20-29,1
1,2,Male,28,Doctor,6.2,6,60,8,Normal,125/80,75,10000,NaN,Male,Doctor,20-29,1
2,3,Male,28,Doctor,6.2,6,60,8,Normal,125/80,75,10000,NaN,Male,Doctor,20-29,1
3,4,Male,28,Sales Representative,5.9,4,30,8,Obese,140/90,85,3000,Sleep Apnea,Male,Sales,20-29,1
4,5,Male,28,Sales Representative,5.9,4,30,8,Obese,140/90,85,3000,Sleep Apnea,Male,Sales,20-29,1


In [9]:
def aggregate_sleep(df: pd.DataFrame) -> pd.DataFrame:
    """Aggregate sleep data by AgeGroup + Gender + Occupation."""
    df = df.copy()

    group_cols = ["AgeGroup", "Gender_std", "Occupation_std"]

    agg = (
        df.groupby(group_cols)
        .agg(
            sl_sleep_duration_mean=("Sleep Duration", "mean"),
            sl_sleep_quality_mean=("Quality of Sleep", "mean"),
            sl_physical_activity_level_mean=("Physical Activity Level", "mean"),
            sl_stress_mean=("Stress Level", "mean"),
            sl_heart_rate_mean=("Heart Rate", "mean"),
            sl_daily_steps_mean=("Daily Steps", "mean"),
            sl_sleep_disorder_rate=("HasSleepDisorder", "mean"),
            n_sleep_records=("Person ID", "count"),
        )
        .reset_index()
    )

    # rename standardized keys for consistency
    agg = agg.rename(columns={"Gender_std": "Gender", "Occupation_std": "Occupation"})

    return agg


# Run aggregation
sleep_agg = aggregate_sleep(sleep_prepared)
sleep_agg.head()


,AgeGroup,Gender,Occupation,sl_sleep_duration_mean,sl_sleep_quality_mean,sl_physical_activity_level_mean,sl_stress_mean,sl_heart_rate_mean,sl_daily_steps_mean,sl_sleep_disorder_rate,n_sleep_records
0,20-29,Female,Nurse,6.50,5.000000,40.0,7.000000,80.000000,4000.000000,1.0,2
1,20-29,Male,Doctor,6.65,6.333333,50.0,7.333333,70.833333,8333.333333,1.0,12
2,20-29,Male,Engineer,6.00,5.000000,36.0,7.000000,81.000000,3600.000000,1.0,2
3,20-29,Male,Sales,5.90,4.000000,30.0,8.000000,85.000000,3000.000000,1.0,2
4,20-29,Male,Teacher,6.30,6.000000,40.0,7.000000,82.000000,3500.000000,1.0,1


In [11]:
# Merge anxiety and sleep aggregated datasets
merged_df = pd.merge(
    anxiety_agg,
    sleep_agg,
    on=["AgeGroup", "Gender", "Occupation"],
    how="inner"
)

merged_df.head()


,AgeGroup,Gender,Occupation,anx_sleep_hours_mean,anx_physical_activity_mean,anx_caffeine_mean,anx_alcohol_mean,anx_stress_mean,anx_heart_rate_mean,anx_diet_quality_mean,...,anx_family_history_rate,n_anxiety_records,sl_sleep_duration_mean,sl_sleep_quality_mean,sl_physical_activity_level_mean,sl_stress_mean,sl_heart_rate_mean,sl_daily_steps_mean,sl_sleep_disorder_rate,n_sleep_records
0,20-29,Female,Nurse,6.683471,2.836364,302.082645,10.000000,5.628099,93.545455,5.272727,...,0.421488,121,6.50,5.000000,40.0,7.000000,80.000000,4000.000000,1.0,2
1,20-29,Male,Doctor,6.642748,2.645038,363.343511,9.709924,5.954198,89.580153,5.190840,...,0.297710,131,6.65,6.333333,50.0,7.333333,70.833333,8333.333333,1.0,12
2,20-29,Male,Engineer,6.863158,2.513158,388.921053,9.807018,5.692982,87.728070,4.500000,...,0.394737,114,6.00,5.000000,36.0,7.000000,81.000000,3600.000000,1.0,2
3,20-29,Male,Teacher,6.753600,2.767200,269.512000,9.496000,5.888000,91.000000,5.080000,...,0.336000,125,6.30,6.000000,40.0,7.000000,82.000000,3500.000000,1.0,1
4,30-39,Female,Lawyer,6.473282,2.461069,383.106870,9.625954,5.900763,93.633588,4.770992,...,0.343511,131,7.15,7.000000,55.0,5.500000,79.500000,4400.000000,1.0,2


In [12]:
def add_feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    """Add engineered features such as sleep_efficiency and risk scores."""
    df = df.copy()

    # 1) Sleep efficiency: relative to 8 hours (ideal)
    df["sleep_efficiency"] = df["sl_sleep_duration_mean"] / 8.0

    # 2) Lifestyle risk score: normalize caffeine, alcohol, smoking, and low sleep
    df["lifestyle_risk_raw"] = (
        (df["anx_caffeine_mean"] / 400.0) +          # 400mg/day ~ high caffeine
        (df["anx_alcohol_mean"] / 14.0) +            # 14 drinks/week ~ high alcohol
        df["anx_smoking_rate"] +                     # smoking proportion
        (1.0 - df["sleep_efficiency"].clip(0, 1))    # low sleep → higher risk
    ) / 4.0

    df["lifestyle_risk_score"] = df["lifestyle_risk_raw"].clip(0, 1)

    # 3) Physical health score: steps + physical activity (two sources)
    df["physical_health_raw"] = (
        (df["sl_daily_steps_mean"] / 10000.0) +              # 10k steps
        (df["sl_physical_activity_level_mean"] / 60.0) +     # 60 min/day
        (df["anx_physical_activity_mean"] / 7.0)             # 7 hrs/week
    ) / 3.0

    df["physical_health_score"] = df["physical_health_raw"].clip(0, 1)

    # 4) High anxiety group indicator: 1 if group high anxiety rate >= 0.4
    df["high_anxiety_group"] = (df["anx_high_anxiety_rate"] >= 0.4).astype(int)

    return df


final_df = add_feature_engineering(merged_df)
final_df.head()


,AgeGroup,Gender,Occupation,anx_sleep_hours_mean,anx_physical_activity_mean,anx_caffeine_mean,anx_alcohol_mean,anx_stress_mean,anx_heart_rate_mean,anx_diet_quality_mean,...,sl_heart_rate_mean,sl_daily_steps_mean,sl_sleep_disorder_rate,n_sleep_records,sleep_efficiency,lifestyle_risk_raw,lifestyle_risk_score,physical_health_raw,physical_health_score,high_anxiety_group
0,20-29,Female,Nurse,6.683471,2.836364,302.082645,10.000000,5.628099,93.545455,5.272727,...,80.000000,4000.000000,1.0,2,0.81250,0.523752,0.523752,0.490620,0.490620,0
1,20-29,Male,Doctor,6.642748,2.645038,363.343511,9.709924,5.954198,89.580153,5.190840,...,70.833333,8333.333333,1.0,12,0.83125,0.528547,0.528547,0.681510,0.681510,0
2,20-29,Male,Engineer,6.863158,2.513158,388.921053,9.807018,5.692982,87.728070,4.500000,...,81.000000,3600.000000,1.0,2,0.75000,0.572806,0.572806,0.439674,0.439674,0
3,20-29,Male,Teacher,6.753600,2.767200,269.512000,9.496000,5.888000,91.000000,5.080000,...,82.000000,3500.000000,1.0,1,0.78750,0.497141,0.497141,0.470660,0.470660,0
4,30-39,Female,Lawyer,6.473282,2.461069,383.106870,9.625954,5.900763,93.633588,4.770992,...,79.500000,4400.000000,1.0,2,0.89375,0.533316,0.533316,0.569416,0.569416,0


In [13]:
# Save final modeling dataset
output_path = DATA_DIR / "final_merged_dataset.csv"

final_df.to_csv(output_path, index=False)

output_path, final_df.shape


(PosixPath('../data/final_merged_dataset.csv'), (20, 29))